# Computational Theory Assignment Problem (SHA-256)


## Problem 1: Representing SHA-256 Data
Before we go ahead with the implementation of SHA-256, I'll first clarify a few things, mainly how input, output, and intermediate data in Python.

NOTE: the small python example I give for each data representation is just a simplified example for demo's sake, the actual implementation is after that.

### 32-bit words
Unlike C/C++/Rust where there's explicit `uint32_t` typing, Python is dynamically typed and its regular integers are not fixed-width. SHA-256 requires every word to behave as an unsigned 32-bit value, so the actual implementation will use NumPy's `np.uint32` type.

Initially, I wanted to go with an `Int32` class wrapper to make it safer, but decided against it because of a few reasons.
1. Performance overhead - SHA-256 produces a 32-bit result in every single operation, and a custom class would create object allocation overhead in the thousands of operations in the function.
2. Lose out on clean bitwise op syntax - a custom class would need many overloaded operators (i.e. `__add__`, `__xor__`, `__rshift__`, etc.), which adds work and more places where things could go wrong.
3. KISS principle, this one is self explanatory

The small example above uses a native Python `int` with a 32-bit mask to demonstrate the idea. In the actual implementation, the reusable `to_uint32` helper applies the mask and converts the result to `np.uint32`, keeping every SHA-256 word within the required 32-bit range.

In [1]:
# this is just for demo, we'll be encapsulating the masking in a helper function later on
word = 0x12345678
word = word & 0xffffffff

print("word in decimal = ", word) # 305419896 (0x12345678 in decimal)
print("word as hexadecimal = ", f"{word:08x}")  # 12345678
# x means hex, 08 means 8 digit, pad if necessary, same format as in C printf

word in decimal =  305419896
word as hexadecimal =  12345678


### sequences of 32-bit words
For the actual implementation, I'll use a mutable NumPy array with `dtype=np.uint32` to represent a sequence of words. This keeps each element fixed at 32 bits and is convenient when the message schedule is mutated during SHA-256.

The earlier `list[int]` example is a simplified demonstration of a word sequence. A Python list is easy to understand, but it does not enforce the 32-bit unsigned representation. The `to_uint32_array` helper below creates the NumPy representation. `bytes` and `bytearray` are alternatives for raw byte data, but word operations would require repeated conversion between bytes and integers.

In [2]:
words = [0x12345678, 0x90abcdef, 0x00000001] # list of 3 words as a demo

print("len(words) = ", len(words))
print([f"{word:08x}" for word in words])

len(words) =  3
['12345678', '90abcdef', '00000001']


### input messages
For the input messages we'll use python's immutable `bytes` type because sha256 operates at a byte level, the `bytes` type is evidently byte-level precise, I'm not going to be using characters because UTF-8 strings can range from 1-4 bytes long e.g. ඞ (0xE0 0xB6 0x9E) is 3 bytes long meanwhile the char `a` is 1 byte long. 

To give further clarification on python's `bytes` type, under the hood, it's literally just a contiguous array of 1 byte values, but with python magic wrapped over it. TLDR it behaves like `const uint8_t[]`. It's also handy because out of the box it provides slicing that's easy to use. An example of the slicing behaviour is in the code below.

I suppose we could theoretically use `str` or `list` here too but that would require extra work. 

e.g. using `str` forces me to encode to bytes before calculating exact bit lengths or applying masks anyway

e.g.2 using `list[int]` of raw byte values wastes memory, loses the super easy byte slicing provided by `bytes` and also require extra conversion steps during the block processing, which is really unnecessary

In [3]:
message = "abc"
message_bytes = message.encode("utf-8")

print("message_bytes = ", message_bytes)  # b'abc', 
# it should actually be the byte val but python tries to be helpful and shows the ascii representation of the bytes if they are printable
print("len(message_bytes) = ", len(message_bytes))  # 3

# slicing example
block = b"\x01\x02\x03\x04\x05\x06\x07\x08\x09\x0A" # this would be an immutable bytes object with 10 bytes
# slicing -> returns a 4-byte 'bytes' object
chunk = block[0:4]   # chunk = b'\x01\x02\x03\x04' (type: bytes)
print("chunk = ", chunk)


message_bytes =  b'abc'
len(message_bytes) =  3
chunk =  b'\x01\x02\x03\x04'


### 512-bit message blocks
Once the input message has been padded, sha256 processes it in 512bit chunks (64 bytes) AKA message blocks. Since the input msg is represented in `bytes`, naturally, each individual block would be a subslice of that big padded input message, meaning they would all be bytes that are 64-byte wide (let's ignore the metadata python attaches to the `bytes` object). And obviously, if there's more than 1 block, I'll be using a standard list ie `list[bytes]`

In [4]:
# initializing an individual block just for demo
block = bytes(range(64))

assert len(block) == 64
assert len(block) * 8 == 512

# in practice it'll be a direct consequence of the padded message, it'll look something like this, which we'll implement later
# I'll explain the padding later too, just know that it's always a multiple of 64 bytes
padded_msg = b"abc" + b"\x80" + b"\x00" * 52 + (24).to_bytes(8, "big")
assert len(padded_msg) == 64

blocks = [
    padded_msg[i : i + 64]
    for i in range(0, len(padded_msg), 64)
]
assert len(blocks) == 1
assert all(len(message_block) == 64 for message_block in blocks)

### the final 256-bit hash value
There's actually 2 useful representation for this, the final hash can be:
1. a 32 bytes for binary ops or storage
2. 64-char hexadecimal string for display (people usually display hashes in this because if we actually printed the 32 raw bytes, it'll be unreadable gibberish since many byte values won't correspond to printable ascii characters)

In [5]:
digest_bytes = bytes.fromhex(
    "ba7816bf8f01cfea414140de5dae2223"
    "b00361a396177a9cb410ff61f20015ad"
)

assert len(digest_bytes) == 32
assert len(digest_bytes) * 8 == 256

print()
print("the raw bytes = ", digest_bytes)  # b'\xba\x78\x16\xbf... you get the idea, but remember the python magic i talked about above, so it might show some ascii when printed
digest_hex = digest_bytes.hex()
print("the hex display = ", digest_hex)  # ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad


the raw bytes =  b'\xbax\x16\xbf\x8f\x01\xcf\xeaAA@\xde]\xae"#\xb0\x03a\xa3\x96\x17z\x9c\xb4\x10\xffa\xf2\x00\x15\xad'
the hex display =  ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad


### big-endian integers
big-endian just means it stores the most significant byte first, e.g. the integer 0x12345678 is stored as:
```
12 34 56 78
```
The endianness matters at three places in SHA-256:
1. at the input boundary: when we split the 64 byte message block (512bit) into 16 initial 32-bit words ($W_0 \dots W_{15}$). The FIPS standard says to interpret each 4-byte group as a big-endian 32-bit integer, so that's what we're following
2. during padding: the original message length in bits is appended as a 64-bit big-endian integer
3. at the output: when we convert the final 32-bit state variables ($H_0 \dots H_7$) back into the final 32-byte hash array, once again just following FIPS, we write out each state integer as 4 big-endian bytes.

I'll show the input and output boundaries in the code below. The padding example above demonstrates the big-endian length field.

NOTE:
inside the main loop, the values are represented as `np.uint32` and the bit shifts, rotations, `add32`, `Ch`, and `Maj` operations do not involve endianness. Endianness only applies when bytes are interpreted as integers or integers are written back into a byte sequence. The conversion helpers handle those byte boundaries.

In [6]:
value = 0x12345678

encoded = value.to_bytes(4, byteorder="big")
decoded = int.from_bytes(encoded, byteorder="big")

print(encoded)  # b'\x124Vx'
print(decoded == value)  # True

b'\x124Vx'
True


### implementation of data types
Below is the setup for the data types used by future problems

In [7]:
# NumPy supplies the fixed-width unsigned integer type required by SHA-256.
import numpy as np

# A 32-bit mask keeps only the least-significant 32 bits of a value.
# This is also how values larger than 0xFFFFFFFF wrap back into the uint32 range.
UINT32_MASK = 0xFFFFFFFF


def to_uint32(value: int | np.integer) -> np.uint32:
    """Convert an integer to a wrapped 32-bit unsigned NumPy value."""
    # Convert NumPy integer scalars to a regular Python integer before masking.
    # Applying the mask explicitly gives predictable modulo-2^32 behavior,
    # including for negative inputs, before constructing the NumPy scalar.
    return np.uint32(int(value) & UINT32_MASK)


def to_uint32_array(values: list[int] | tuple[int, ...]) -> np.ndarray:
    """Convert a sequence of integers into a mutable uint32 NumPy array."""
    # Normalize each input through the shared helper so every element is
    # guaranteed to have the same fixed-width representation.
    # The dtype argument makes the array's element type explicit rather than
    # relying on NumPy to infer it from the input values.
    return np.array([to_uint32(value) for value in values], dtype=np.uint32)


def block_to_words(block: bytes) -> np.ndarray:
    """Decode one 64-byte, big-endian SHA-256 block into sixteen uint32 words."""
    # SHA-256 processes exactly 512 bits, which is 64 bytes, per block.
    # Rejecting other lengths prevents silently decoding malformed input.
    if len(block) != 64:
        raise ValueError("a SHA-256 block must contain exactly 64 bytes")

    # The >u4 dtype means big-endian, unsigned 4-byte integers. This matches
    # the SHA-256 standard, which interprets every group of four input bytes
    # as one big-endian 32-bit word.
    # astype converts the result to NumPy's explicit uint32 dtype for the
    # bitwise operations that will be performed later.
    return np.frombuffer(block, dtype=">u4").astype(np.uint32)


def words_to_bytes(words: list[int] | tuple[int, ...] | np.ndarray) -> bytes:
    """Encode uint32 words as a big-endian byte sequence."""
    # Re-normalize the values so this boundary also handles ordinary Python
    # integers and guarantees that every value fits in 32 bits.
    uint32_words = to_uint32_array(list(words))

    # Convert each word to a big-endian four-byte representation, then expose
    # the resulting contiguous byte data as an immutable Python bytes object.
    return uint32_words.astype(">u4").tobytes()


# These examples demonstrate the representations used by the helpers.
word = to_uint32(0x12345678)
words = to_uint32_array([0x12345678, 0x90ABCDEF, 0x00000001])
block = bytes(range(64))
block_words = block_to_words(block)

# Confirm that individual words and decoded block words use NumPy uint32.
assert isinstance(word, np.uint32)
assert words.dtype == np.uint32
assert len(block_words) == 16
assert all(isinstance(value, np.uint32) for value in block_words)

# Confirm that the output boundary writes a word in big-endian order.
assert words_to_bytes([0x12345678]) == bytes.fromhex("12345678")

print(f"word = {word:08x} ({type(word).__name__})")
print(f"words dtype = {words.dtype}")
print(f"block contains {len(block_words)} uint32 words")

word = 12345678 (uint32)
words dtype = uint32
block contains 16 uint32 words


### Tests for the data type helpers (just in case)
The examples above demonstrate typical usage. The tests below verify the required types, 32-bit wrapping, mutability, big-endian conversion, validation, and round-trip behavior. I know I already added assertions above but this will might save me some headache down the line

In [8]:
# unittest is part of Python's standard library, so no extra test dependency
# is needed for this notebook.
import unittest


class TestSha256DataTypes(unittest.TestCase):
    def test_to_uint32_returns_uint32_and_wraps_values(self):
        # A normal word should be represented by NumPy's fixed-width uint32
        # scalar rather than by an unrestricted Python integer.
        self.assertIsInstance(to_uint32(0x12345678), np.uint32)

        # The helper must discard bits above bit 31, which is equivalent to
        # taking the input modulo 2^32.
        self.assertEqual(int(to_uint32(0x1_0000_0001)), 0x00000001)

        # Masking a negative Python integer produces its two's-complement
        # 32-bit representation, so -1 becomes all 32 bits set.
        self.assertEqual(int(to_uint32(-1)), UINT32_MASK)

    def test_to_uint32_array_returns_mutable_uint32_array(self):
        # The sequence helper should create the mutable array representation
        # used for word sequences and message schedules.
        values = to_uint32_array([0x12345678, 0x90ABCDEF])

        # Check both the container type and its element dtype. Checking dtype
        # matters because a generic integer array would not satisfy the SHA-256
        # fixed-width representation requirement.
        self.assertIsInstance(values, np.ndarray)
        self.assertEqual(values.dtype, np.dtype(np.uint32))
        self.assertEqual(values.tolist(), [0x12345678, 0x90ABCDEF])

        # A message schedule is updated in place, so verify that the returned
        # array is genuinely mutable rather than an immutable sequence.
        values[0] = to_uint32(0xCAFEBABE)
        self.assertEqual(int(values[0]), 0xCAFEBABE)

    def test_block_to_words_decodes_big_endian_words(self):
        # bytes(range(64)) gives a deterministic 64-byte block whose byte
        # ordering makes the expected first and last words easy to calculate.
        block = bytes(range(64))
        words = block_to_words(block)

        # A valid block must become sixteen uint32 words because 64 / 4 = 16.
        self.assertEqual(words.dtype, np.dtype(np.uint32))
        self.assertEqual(len(words), 16)

        # The first four bytes are 00 01 02 03 and must be read as the
        # big-endian integer 0x00010203.
        self.assertEqual(int(words[0]), 0x00010203)

        # The final four bytes are 3C 3D 3E 3F and must be read as
        # 0x3C3D3E3F, confirming the end of the block is decoded correctly.
        self.assertEqual(int(words[-1]), 0x3C3D3E3F)

    def test_block_to_words_rejects_invalid_block_length(self):
        # SHA-256 compression works on exactly 512-bit blocks, so malformed
        # input should fail explicitly instead of producing partial words.
        with self.assertRaises(ValueError):
            block_to_words(b"too short")

    def test_words_to_bytes_encodes_big_endian_words(self):
        # These words make the byte order visible: each four-byte group should
        # appear most-significant byte first in the output.
        words = [0x12345678, 0x90ABCDEF]

        self.assertEqual(
            words_to_bytes(words),
            bytes.fromhex("1234567890abcdef"),
        )

    def test_block_word_conversion_round_trips(self):
        # Converting a valid block to words and back should preserve every byte.
        # This checks both conversion helpers together at their boundary.
        block = bytes(range(64))

        self.assertEqual(words_to_bytes(block_to_words(block)), block)


# Discover the test methods on the test class and execute them with readable
# per-test output. The final assertion makes the notebook cell fail if any
# test fails, which is useful when running the notebook top to bottom.
result = unittest.TextTestRunner(verbosity=2).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(TestSha256DataTypes)
)
assert result.wasSuccessful()

test_block_to_words_decodes_big_endian_words (__main__.TestSha256DataTypes.test_block_to_words_decodes_big_endian_words) ... ok
test_block_to_words_rejects_invalid_block_length (__main__.TestSha256DataTypes.test_block_to_words_rejects_invalid_block_length) ... ok
test_block_word_conversion_round_trips (__main__.TestSha256DataTypes.test_block_word_conversion_round_trips) ... ok
test_to_uint32_array_returns_mutable_uint32_array (__main__.TestSha256DataTypes.test_to_uint32_array_returns_mutable_uint32_array) ... ok
test_to_uint32_returns_uint32_and_wraps_values (__main__.TestSha256DataTypes.test_to_uint32_returns_uint32_and_wraps_values) ... ok
test_words_to_bytes_encodes_big_endian_words (__main__.TestSha256DataTypes.test_words_to_bytes_encodes_big_endian_words) ... ok

----------------------------------------------------------------------
Ran 6 tests in 0.010s

OK


## Problem 2: SHA-256 Bitwise Operations

Before we go into actually implementing the functions, let's go through a big picture explanation first, so we can see where the functions we are going to implement like Ch() and Maj() comes into play in the big picture:

SHA-256 takes input messages and turns them into a fixed 256-bit signature through iterative mixing. To make the output completely unpredictable and impossible to reverse, sha256 breaks every message into 512bit blocks, expands each block into a 64-word schedule, and runs 64 compression rounds over eight state variables ($a, b, c, d, e, f, g, h$).

The six functions in Problem 2's task are the elementary mixing building blocks used inside those 64 rounds:
- The Sigmas ($\sigma_0, \sigma_1, \Sigma_0, \Sigma_1$) act as diffusion engines. They take a word, rotate its bits by different distances, and XOR them together. This ensures that altering even a single bit in your input completely scrambles every bit in future steps AKA the avalanche effect.
- Ch and Maj act as non-linear logic gates. They combine bits from three different state variables so that an attacker cannot solve for past inputs using simple linear algebra. (kinda like how activation functions in ML NN stops the whole network from collapsing into a single linear alg equation, they do the same job here, provide non linearity)

### `Ch(x,y,z)`
Ch means Choose. For each bit:
If the corresponding bit in x is 1, choose the bit from y.
If the corresponding bit in x is 0, choose the bit from z.

It's easier to see with an example so that's what I'm going to be showing for each function.

#### Example
Using 4-bit values:
```text
x = 1010
y = 1100
z = 0011
```
Calculate:
```text
x AND y       = 1000
NOT x AND z   = 0101 AND 0011 = 0001
1000 XOR 0001 = 1001
```
Therefore:
```text
Ch(1010, 1100, 0011) = 1001
```
The result selects bits from y where x has `1`s, and bits from z where x has `0`s, that's all there is to it

#### implementation for `Ch(x, y, z)`

In [9]:
def Ch(x: np.uint32, y: np.uint32, z: np.uint32) -> np.uint32:
    """Choose bits from y or z according to the corresponding bits in x."""
    # Convert NumPy scalars to Python integers before applying the bitwise
    # operations so the complement can be explicitly limited to 32 bits.
    x_value = int(x)
    y_value = int(y)
    z_value = int(z)

    # For each bit, x selects y when it is 1 and z when it is 0:
    # Ch(x, y, z) = (x AND y) XOR (NOT x AND z).
    selected_y_bits = x_value & y_value
    selected_z_bits = (~x_value & UINT32_MASK) & z_value

    # Reuse the shared helper so the result is wrapped and returned as a
    # NumPy uint32, matching the representation used throughout SHA-256.
    return to_uint32(selected_y_bits ^ selected_z_bits)

#### unit test for `Ch(x, y, z)`

In [10]:
# Unit tests for Ch(x, y, z)
import unittest

import numpy as np


class TestCh(unittest.TestCase):
    def test_ch_selects_y_when_x_bit_is_one_and_z_when_x_bit_is_zero(self):
        # Ch is a bit-by-bit selector. For every bit position, x determines
        # whether the result receives the bit from y or from z.
        x = np.uint32(0b1010)
        y = np.uint32(0b1100)
        z = np.uint32(0b0011)

        # The expected result is 1001:
        # - x has 1 in the left and right positions, so those bits come from y.
        # - x has 0 in the other positions, so those bits come from z.
        expected = np.uint32(0b1001)

        actual = Ch(x, y, z)

        # Checking the value verifies the Boolean definition:
        # Ch(x, y, z) = (x AND y) XOR (NOT x AND z).
        self.assertEqual(actual, expected)

        # SHA-256 operates on fixed-width unsigned words, so the result should
        # remain a NumPy uint32 value rather than becoming an unrestricted int.
        self.assertIsInstance(actual, np.uint32)

    def test_ch_returns_y_when_x_contains_only_ones(self):
        # When x is all ones, every bit position selects the corresponding bit
        # from y. The z value should have no effect on the result.
        x = np.uint32(0xFFFFFFFF)
        y = np.uint32(0x12345678)
        z = np.uint32(0xAAAAAAAA)

        actual = Ch(x, y, z)

        # This confirms the selector behavior for all 32 bit positions at once.
        self.assertEqual(actual, y)
        self.assertIsInstance(actual, np.uint32)

    def test_ch_returns_z_when_x_contains_only_zeros(self):
        # When x is all zeros, every bit position selects the corresponding bit
        # from z. The y value should have no effect on the result.
        x = np.uint32(0x00000000)
        y = np.uint32(0x12345678)
        z = np.uint32(0xAAAAAAAA)

        actual = Ch(x, y, z)

        # This checks the opposite extreme case from the previous test and
        # verifies that all 32 positions select z correctly.
        self.assertEqual(actual, z)
        self.assertIsInstance(actual, np.uint32)

    def test_ch_uses_the_sha256_bitwise_formula(self):
        # These values contain a mixture of zero and one bits so both branches
        # of the selector are exercised throughout the 32-bit word.
        x = np.uint32(0xAAAAAAAA)
        y = np.uint32(0xCCCCCCCC)
        z = np.uint32(0x33333333)

        # Calculate the expected value directly from the SHA-256 definition.
        # The explicit mask keeps the NOT operation within 32 bits, matching
        # SHA-256's fixed-width word behavior.
        expected = np.uint32(
            (int(x) & int(y)) ^ ((~int(x) & 0xFFFFFFFF) & int(z))
        )

        actual = Ch(x, y, z)

        # Comparing with an independently calculated expected value helps catch
        # incorrect operand ordering or a missing 32-bit mask in the function.
        self.assertEqual(actual, expected)
        self.assertIsInstance(actual, np.uint32)


# Run the Ch tests with readable per-test output. The final assertion makes
# this cell fail if any test fails.
result = unittest.TextTestRunner(verbosity=2).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(TestCh)
)
assert result.wasSuccessful()

test_ch_returns_y_when_x_contains_only_ones (__main__.TestCh.test_ch_returns_y_when_x_contains_only_ones) ... ok
test_ch_returns_z_when_x_contains_only_zeros (__main__.TestCh.test_ch_returns_z_when_x_contains_only_zeros) ... ok
test_ch_selects_y_when_x_bit_is_one_and_z_when_x_bit_is_zero (__main__.TestCh.test_ch_selects_y_when_x_bit_is_one_and_z_when_x_bit_is_zero) ... ok
test_ch_uses_the_sha256_bitwise_formula (__main__.TestCh.test_ch_uses_the_sha256_bitwise_formula) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.010s

OK


### `Maj(x, y, z)`

Maj means majority. For each bit position, it returns the bit that appears most often among x,y,z, it's a pretty straight forward one.

#### Example
Using 4-bit values (same as Ch for simplicity sake):
```text
x = 1010
y = 1100
z = 0011
```
Calculate:
```text
x AND y = 1000
x AND z = 0010
y AND z = 0000

then 

1000 XOR 0010 XOR 0000 = 1010
```
Therefore:
```text
Maj(1010, 1100, 0011) = 1010
```
Basically, each res bit is just the one that appears the most between x y and z. 

#### Implementation for `Maj(x, y, z)`

In [11]:
def Maj(x: np.uint32, y: np.uint32, z: np.uint32) -> np.uint32:
    """Return the bitwise majority value of three uint32 words."""
    # Convert NumPy scalars to Python integers before combining the pairwise
    # results, then normalize the final value back to uint32.
    x_value = int(x)
    y_value = int(y)
    z_value = int(z)

    # A bit is set in the majority result when it is present in at least two
    # inputs. The three pairwise AND terms capture those possible agreements.
    pairwise_majority = (
        (x_value & y_value)
        ^ (x_value & z_value)
        ^ (y_value & z_value)
    )

    # Use the shared helper so the result has the fixed-width representation
    # required by the rest of the SHA-256 implementation.
    return to_uint32(pairwise_majority)

#### Unit test for `Maj(x, y, z)`

In [12]:
import unittest

import numpy as np


class TestMaj(unittest.TestCase):
    def test_maj_returns_the_bitwise_majority(self):
        # Maj examines each bit position independently and returns the bit that
        # appears at least twice among x, y, and z.
        x = np.uint32(0b1010)
        y = np.uint32(0b1100)
        z = np.uint32(0b0011)

        # The pairwise AND values identify positions where each pair agrees:
        # x AND y = 1000, x AND z = 0010, and y AND z = 0000.
        expected = np.uint32(0b1010)

        actual = Maj(x, y, z)

        # This checks the concrete four-bit example from the explanation.
        self.assertEqual(actual, expected)
        self.assertIsInstance(actual, np.uint32)

    def test_maj_with_all_one_x_selects_the_union_of_y_and_z(self):
        # If x contains only ones, each result bit is one whenever either y or z
        # contains one, so the result should be y OR z.
        x = np.uint32(0xFFFFFFFF)
        y = np.uint32(0x12345678)
        z = np.uint32(0xAAAAAAAA)

        expected = np.uint32(int(y) | int(z))
        actual = Maj(x, y, z)

        self.assertEqual(actual, expected)
        self.assertIsInstance(actual, np.uint32)

    def test_maj_with_all_zero_x_selects_the_intersection_of_y_and_z(self):
        # If x contains only zeros, a result bit can be one only when both y
        # and z contain one at that same position, so the result should be y AND z.
        x = np.uint32(0x00000000)
        y = np.uint32(0x12345678)
        z = np.uint32(0xAAAAAAAA)

        expected = np.uint32(int(y) & int(z))
        actual = Maj(x, y, z)

        self.assertEqual(actual, expected)
        self.assertIsInstance(actual, np.uint32)

    def test_maj_matches_the_sha256_bitwise_formula(self):
        # These values exercise many different combinations of zero and one
        # bits throughout the 32-bit word.
        x = np.uint32(0xAAAAAAAA)
        y = np.uint32(0xCCCCCCCC)
        z = np.uint32(0x33333333)

        # Calculate the expected result independently from the standard formula:
        # Maj(x, y, z) = (x AND y) XOR (x AND z) XOR (y AND z).
        expected = np.uint32(
            (int(x) & int(y)) ^ (int(x) & int(z)) ^ (int(y) & int(z))
        )

        actual = Maj(x, y, z)

        self.assertEqual(actual, expected)
        self.assertIsInstance(actual, np.uint32)


result = unittest.TextTestRunner(verbosity=2).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(TestMaj)
)
assert result.wasSuccessful()

test_maj_matches_the_sha256_bitwise_formula (__main__.TestMaj.test_maj_matches_the_sha256_bitwise_formula) ... ok
test_maj_returns_the_bitwise_majority (__main__.TestMaj.test_maj_returns_the_bitwise_majority) ... ok
test_maj_with_all_one_x_selects_the_union_of_y_and_z (__main__.TestMaj.test_maj_with_all_one_x_selects_the_union_of_y_and_z) ... ok
test_maj_with_all_zero_x_selects_the_intersection_of_y_and_z (__main__.TestMaj.test_maj_with_all_zero_x_selects_the_intersection_of_y_and_z) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.015s

OK


### `Sigma0(x)` $\Sigma^0_{256}(x)$

#### Implementation for `Sigma0(x)` $\Sigma^0_{256}(x)$

In [25]:
def Sigma0(x: np.uint32) -> np.uint32:
    """Return the SHA-256 uppercase Sigma0 transformation."""
    x_value = int(x)
    rotate_right_2 = ((x_value >> 2) | (x_value << 30)) & UINT32_MASK
    rotate_right_13 = ((x_value >> 13) | (x_value << 19)) & UINT32_MASK
    rotate_right_22 = ((x_value >> 22) | (x_value << 10)) & UINT32_MASK
    return to_uint32(rotate_right_2 ^ rotate_right_13 ^ rotate_right_22)

#### Unit test for `Sigma0(x)` $\Sigma^0_{256}(x)$

In [26]:
import unittest

import numpy as np


class TestSigma0(unittest.TestCase):
    def test_sigma0_matches_sha256_reference_vectors(self):
        test_cases = [
            (0x12345678, 0x66146474),
            (0x00000001, 0x40080400),
            (0x80000000, 0x20040200),
            (0xFFFFFFFF, 0xFFFFFFFF),
        ]

        for input_value, expected_value in test_cases:
            with self.subTest(input_value=f"0x{input_value:08x}"):
                actual = Sigma0(np.uint32(input_value))

                self.assertEqual(actual, np.uint32(expected_value))
                self.assertIsInstance(actual, np.uint32)


result = unittest.TextTestRunner(verbosity=2).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(TestSigma0)
)
assert result.wasSuccessful()

test_sigma0_matches_sha256_reference_vectors (__main__.TestSigma0.test_sigma0_matches_sha256_reference_vectors) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.004s

OK


### `Sigma1(x)` $\Sigma^1_{256}(x)$

#### Implementation for `Sigma1(x)` $\Sigma^1_{256}(x)$

In [27]:
def Sigma1(x: np.uint32) -> np.uint32:
    """Return the SHA-256 uppercase Sigma1 transformation."""
    x_value = int(x)
    rotate_right_6 = ((x_value >> 6) | (x_value << 26)) & UINT32_MASK
    rotate_right_11 = ((x_value >> 11) | (x_value << 21)) & UINT32_MASK
    rotate_right_25 = ((x_value >> 25) | (x_value << 7)) & UINT32_MASK
    return to_uint32(rotate_right_6 ^ rotate_right_11 ^ rotate_right_25)

#### Unit test for `Sigma1(x)` $\Sigma^1_{256}(x)$

In [28]:
import unittest

import numpy as np


class TestSigma1(unittest.TestCase):
    def test_sigma1_matches_sha256_reference_vectors(self):
        test_cases = [
            (0x12345678, 0x3561ABDA),
            (0x00000001, 0x04200080),
            (0x80000000, 0x02100040),
            (0xFFFFFFFF, 0xFFFFFFFF),
        ]

        for input_value, expected_value in test_cases:
            with self.subTest(input_value=f"0x{input_value:08x}"):
                actual = Sigma1(np.uint32(input_value))

                self.assertEqual(actual, np.uint32(expected_value))
                self.assertIsInstance(actual, np.uint32)


result = unittest.TextTestRunner(verbosity=2).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(TestSigma1)
)
assert result.wasSuccessful()

test_sigma1_matches_sha256_reference_vectors (__main__.TestSigma1.test_sigma1_matches_sha256_reference_vectors) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.005s

OK


### `sigma0(x)` $\sigma^0_{256}(x)$

#### Implementation for `sigma0(x)` $\sigma^0_{256}(x)$

In [17]:
def sigma0(x: np.uint32) -> np.uint32:
    """Temporarily reserve the SHA-256 lowercase sigma0 operation."""
    raise NotImplementedError("sigma0 is not implemented yet")

#### Unit test for `sigma0(x)` $\sigma^0_{256}(x)$

In [23]:
import unittest

import numpy as np


class Testsigma0(unittest.TestCase):
    def test_sigma0_matches_sha256_reference_vectors(self):
        test_cases = [
            (0x12345678, 0xE7FCE6EE),
            (0x00000001, 0x02004000),
            (0x80000000, 0x11002000),
            (0xFFFFFFFF, 0x1FFFFFFF),
        ]

        for input_value, expected_value in test_cases:
            with self.subTest(input_value=f"0x{input_value:08x}"):
                actual = sigma0(np.uint32(input_value))

                self.assertEqual(actual, np.uint32(expected_value))
                self.assertIsInstance(actual, np.uint32)


result = unittest.TextTestRunner(verbosity=2).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(Testsigma0)
)
assert result.wasSuccessful()

test_sigma0_matches_sha256_reference_vectors (__main__.Testsigma0.test_sigma0_matches_sha256_reference_vectors) ... 
  test_sigma0_matches_sha256_reference_vectors (__main__.Testsigma0.test_sigma0_matches_sha256_reference_vectors) (input_value='0x12345678') ... ERROR
  test_sigma0_matches_sha256_reference_vectors (__main__.Testsigma0.test_sigma0_matches_sha256_reference_vectors) (input_value='0x00000001') ... ERROR
  test_sigma0_matches_sha256_reference_vectors (__main__.Testsigma0.test_sigma0_matches_sha256_reference_vectors) (input_value='0x80000000') ... ERROR
  test_sigma0_matches_sha256_reference_vectors (__main__.Testsigma0.test_sigma0_matches_sha256_reference_vectors) (input_value='0xffffffff') ... ERROR

ERROR: test_sigma0_matches_sha256_reference_vectors (__main__.Testsigma0.test_sigma0_matches_sha256_reference_vectors) (input_value='0x12345678')
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/tmp/ipykernel_108

AssertionError: 

### `sigma1(x)` $\sigma^1_{256}(x)$

#### Implementation for `sigma1(x)` $\sigma^1_{256}(x)$

In [19]:
def sigma1(x: np.uint32) -> np.uint32:
    """Temporarily reserve the SHA-256 lowercase sigma1 operation."""
    raise NotImplementedError("sigma1 is not implemented yet")

#### Unit test for `sigma1(x)` $\sigma^1_{256}(x)$

In [24]:
import unittest

import numpy as np


class Testsigma1(unittest.TestCase):
    def test_sigma1_matches_sha256_reference_vectors(self):
        test_cases = [
            (0x12345678, 0xA1F78649),
            (0x00000001, 0x0000A000),
            (0x80000000, 0x00205000),
            (0xFFFFFFFF, 0x003FFFFF),
        ]

        for input_value, expected_value in test_cases:
            with self.subTest(input_value=f"0x{input_value:08x}"):
                actual = sigma1(np.uint32(input_value))

                self.assertEqual(actual, np.uint32(expected_value))
                self.assertIsInstance(actual, np.uint32)


result = unittest.TextTestRunner(verbosity=2).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(Testsigma1)
)
assert result.wasSuccessful()

test_sigma1_matches_sha256_reference_vectors (__main__.Testsigma1.test_sigma1_matches_sha256_reference_vectors) ... 
  test_sigma1_matches_sha256_reference_vectors (__main__.Testsigma1.test_sigma1_matches_sha256_reference_vectors) (input_value='0x12345678') ... ERROR
  test_sigma1_matches_sha256_reference_vectors (__main__.Testsigma1.test_sigma1_matches_sha256_reference_vectors) (input_value='0x00000001') ... ERROR
  test_sigma1_matches_sha256_reference_vectors (__main__.Testsigma1.test_sigma1_matches_sha256_reference_vectors) (input_value='0x80000000') ... ERROR
  test_sigma1_matches_sha256_reference_vectors (__main__.Testsigma1.test_sigma1_matches_sha256_reference_vectors) (input_value='0xffffffff') ... ERROR

ERROR: test_sigma1_matches_sha256_reference_vectors (__main__.Testsigma1.test_sigma1_matches_sha256_reference_vectors) (input_value='0x12345678')
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/tmp/ipykernel_108

AssertionError: 